# Evaluate Climate Models

### Parameter Settings: Change Timestep here

In [ ]:
# %% [Setup — Distribution Difference]

import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"

import sys
from pathlib import Path

HELPER_DIR = Path("/nird/home/lbal/internship_storm_hans/helper")
if str(HELPER_DIR) not in sys.path:
    sys.path.insert(0, str(HELPER_DIR))

import config_paths as cfg

# ── Define the two figure-output root directories ─────────────────────────────
FIG_SUBDIR = "climate_model_evaluation"
FIG_DIRS = [
    cfg.FIGURES_DIR         / FIG_SUBDIR,
    cfg.FIGURES_DIR_SECONDARY / FIG_SUBDIR,
]
for d in FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)

from data_smile import get_year_range_smile

from catchment_tools import (
    load_smile_annual_maxima_per_year,
    load_smile_daily_values_per_year,        # NEW
    load_reanalysis_values_per_year,         # NEW
    compute_distribution_difference,
)
from plot_style import make_distribution_difference_figure

# ── Output: subfolder "distribution_difference" inside each FIG_DIR ──────────
DIST_DIFF_FIG_DIRS = [
    d / "distribution_difference" for d in FIG_DIRS
]
for d in DIST_DIFF_FIG_DIRS:
    d.mkdir(parents=True, exist_ok=True)

# All (window_before, window_after) combinations.
# No year is ever used in two different pairs.
DIST_DIFF_COMBOS = [
    (1,  1),
    (2,  1),  (2,  2),
    (5,  1),  (5,  2),  (5,  5),
    (10, 1),  (10, 2),  (10, 10),
    (30, 1),  (30, 2),  (30, 5),  (30, 10), (30, 30),
]

BIN_WIDTH_MM = 2.5

# ── SMILE datasets: full available period ─────────────────────────────────────
SMILE_DATASETS_DD = {}
for ds_key, smile_entry in cfg.SMILE_CONFIG.items():
    avail_start, avail_end = get_year_range_smile(smile_entry["model_dir"], ds_key)
    SMILE_DATASETS_DD[ds_key] = {
        "start_year": avail_start,
        "end_year":   2024,
    }
    print(f"  [SMILE] {ds_key}: {avail_start}–2024 "
          f"({2024 - avail_start + 1} yr available)")

# ── Reanalysis datasets: label → (dataset, resolution) ───────────────────────
REANALYSIS_DD = [
    ("senorge",   "senorge", ""),
    ("era5_0.25", "era5",    "0.25x0.25"),
    ("era5_0.5",  "era5",    "0.5x0.5"),
]

print(f"\nBin width : {BIN_WIDTH_MM} mm")
print(f"Combos    : {len(DIST_DIFF_COMBOS)}")
print(f"Saving to : {DIST_DIFF_FIG_DIRS[0]}")

In [ ]:
# %% [Load per-catchment data for evaluation figures]
#
# Loads annual maxima and daily values per catchment, per model, for the
# evaluation period. Results are used by:
#   - make_distribution_figure  (distribution analysis plots)
#   - make_qq_figure            (Q-Q mapping plots)
#   - build_percentile_mapping_table / build_distribution_summary_table

from catchment_tools import (
    load_annual_maxima_per_catchment,
    load_daily_values_per_catchment,
    build_percentile_mapping_table,
    build_distribution_summary_table,
)
from plot_style import make_distribution_figure, make_qq_figure

# ── Evaluation period ─────────────────────────────────────────────────────────
# Must be contained within both the SMILE and reanalysis cached ranges.
# Adjust if your cached data uses a different overlap period.
EVAL_START_YEAR = 1985
EVAL_END_YEAR   = 2024
EVAL_PERIOD_TAG = f"{EVAL_START_YEAR}-{EVAL_END_YEAR}"

REANALYSIS_KEYS     = ["senorge", "era5_0.25", "era5_0.5"]
PERCENTILES_TO_COMPARE = (2.5, 5, 16, 50, 84, 95, 97.5)

# ── Load annual maxima per catchment ─────────────────────────────────────────
print("Loading 1-day annual maxima per catchment ...")
am_1day_per_catchment = load_annual_maxima_per_catchment(
    window_days = 1,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

print("Loading 2-day annual maxima per catchment ...")
am_2day_per_catchment = load_annual_maxima_per_catchment(
    window_days = 2,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

# ── Load daily values per catchment ──────────────────────────────────────────
print("Loading 1-day daily values per catchment ...")
daily_1day_per_catchment = load_daily_values_per_catchment(
    window_days = 1,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

print("Loading 2-day daily values per catchment ...")
daily_2day_per_catchment = load_daily_values_per_catchment(
    window_days = 2,
    start_year  = EVAL_START_YEAR,
    end_year    = EVAL_END_YEAR,
)

print(f"\n✓ Done. Eval period: {EVAL_PERIOD_TAG}")
print(f"  Catchments : {list(cfg.CATCHMENTS.keys())}")
for slug in cfg.CATCHMENTS:
    models_1day = list(am_1day_per_catchment[slug].keys())
    print(f"  {slug}: {models_1day}")

### Create Distribution plots

In [ ]:
# %% [Distribution Analysis — per catchment, per window, annual_max + daily]
#
# Produces one PDF per (catchment × window_days × data_type) combination.
#
# Filename pattern:
#   {data_type}_distribution_{window_days}day_{slug}_{start}-{end}.pdf
#
# Saved to: FIG_DIRS (climate_model_evaluation/, NO subfolder)

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc, daily_pc in [
        (1, am_1day_per_catchment, daily_1day_per_catchment),
        (2, am_2day_per_catchment, daily_2day_per_catchment),
    ]:
        for data_type, pc in [
            ("annual_max", am_pc),
            ("daily",      daily_pc),
        ]:
            data_for_catchment = pc[slug]

            if not data_for_catchment:
                print(f"  [skip] No data for {slug} / {window_days}day / {data_type}")
                continue

            fname = (
                f"{data_type}_distribution_{window_days}day_"
                f"{slug}_{EVAL_PERIOD_TAG}.pdf"
            )
            out_paths = [d / fname for d in FIG_DIRS]

            print(f"  Saving: {fname}")
            make_distribution_figure(
                annual_maxima   = data_for_catchment,
                window_days     = window_days,
                out_paths       = out_paths,
                data_type       = data_type,
                catchment_title = catchment_title,
            )

print(f"\n[distribution] ✓ Done. Figures saved to:")
for d in FIG_DIRS:
    print(f"  {d}")

### Create Distribution Difference plots

In [ ]:
# %% [Distribution Difference — main loop: SMILE + reanalysis, annual max + daily]
#
# Iterates over:
#   sources   : all SMILE datasets + all three reanalysis datasets
#   data_types: "annual_max"  and  "daily"
#   window_days: 1-day and 2-day accumulation
#   catchments: all cfg.CATCHMENTS
#   combos    : all (wb, wa) in DIST_DIFF_COMBOS
#
# Output filenames:
#   annual_max_distdiff_{wb}year_{wa}year_{dataset}_{wday}day_{slug}_{start}-{end}.pdf
#   daily_distdiff_{wb}year_{wa}year_{dataset}_{wday}day_{slug}_{start}-{end}.pdf
#
# First run may be slow for SMILE (raw member caches built on demand).
# Subsequent runs are fast (all caches reused).

# ── Helper: single source × data_type × window_days × catchment run ──────────

def _run_distdiff_for_source(
    *,
    dataset_key: str,        # model key, e.g. "cesm2_le" or "era5_0.5"
    dataset_arg: str,        # cfg dataset name, e.g. "era5"
    resolution: str,         # e.g. "0.25x0.25" or ""
    is_smile: bool,
    ds_start: int,
    ds_end: int,
    window_days: int,
    data_type: str,          # "annual_max" or "daily"
    slug: str,
    catchment_title: str,
    fig_dirs: list,
    dist_diff_combos: list,
    bin_width_mm: float,
    force_recompute: bool = False,
) -> None:
    """Load per-year data for one source and run all combo figures."""

    n_years = ds_end - ds_start + 1
    type_tag = "annual_max" if data_type == "annual_max" else "daily"

    # ── Load per-year data ────────────────────────────────────────────────────
    if is_smile:
        if data_type == "annual_max":
            per_year = load_smile_annual_maxima_per_year(
                dataset        = dataset_arg,
                window_days    = window_days,
                catchment_slug = slug,
                start_year     = ds_start,
                end_year       = ds_end,
                force_recompute= force_recompute,
            )
        else:
            per_year = load_smile_daily_values_per_year(
                dataset        = dataset_arg,
                window_days    = window_days,
                catchment_slug = slug,
                start_year     = ds_start,
                end_year       = ds_end,
                force_recompute= force_recompute,
            )
    else:
        per_year = load_reanalysis_values_per_year(
            dataset        = dataset_arg,
            resolution     = resolution,
            window_days    = window_days,
            catchment_slug = slug,
            start_year     = ds_start,
            end_year       = ds_end,
            data_type      = data_type,
        )

    if not per_year:
        print(f"    [skip] No per-year data for {dataset_key}/{slug}")
        return

    n_members_check = next(iter(per_year.values())).size
    print(f"    Loaded: {min(per_year)}–{max(per_year)}  "
          f"({len(per_year)} years,  {n_members_check} values/yr)")

    # ── Run all (wb, wa) combos ───────────────────────────────────────────────
    for wb, wa in dist_diff_combos:
        total_needed = wb + wa
        if total_needed > n_years:
            print(f"    [skip] {wb}yr→{wa}yr : need {total_needed} yr "
                  f"but only {n_years} available")
            continue

        bin_centers, avg_diff, n_pairs = compute_distribution_difference(
            per_year_data = per_year,
            window_before = wb,
            window_after  = wa,
            bin_width_mm  = bin_width_mm,
        )
        if n_pairs == 0:
            print(f"    [skip] {wb}yr→{wa}yr : 0 non-overlapping pairs")
            continue

        fname = (
            f"{type_tag}_distdiff_{wb}year_{wa}year_"
            f"{dataset_key}_{window_days}day_"
            f"{slug}_{ds_start}-{ds_end}.pdf"
        )
        out_paths = [d / fname for d in fig_dirs]

        make_distribution_difference_figure(
            bin_centers     = bin_centers,
            avg_diff        = avg_diff,
            n_pairs         = n_pairs,
            window_before   = wb,
            window_after    = wa,
            dataset         = dataset_key,
            window_days     = window_days,
            catchment_title = catchment_title,
            start_year      = ds_start,
            end_year        = ds_end,
            out_paths       = out_paths,
            bin_width_mm    = bin_width_mm,
            data_type       = data_type,)
        print(f"    [ok]   {type_tag} {wb}yr→{wa}yr  n_pairs={n_pairs}")


# ── Main loop ─────────────────────────────────────────────────────────────────

for window_days in (1, 2):
    print(f"\n{'='*70}")
    print(f"[distdiff]  window_days = {window_days}")
    print(f"{'='*70}")

    # ── SMILE datasets ────────────────────────────────────────────────────────
    for ds_key, ds_info in SMILE_DATASETS_DD.items():
        ds_start = ds_info["start_year"]
        ds_end   = ds_info["end_year"]
        for data_type in ("annual_max", "daily"):
            print(f"\n  [SMILE] {ds_key} | {data_type} | {ds_start}–{ds_end}")
            for slug, catchment_title in cfg.CATCHMENTS.items():
                print(f"  Catchment: {catchment_title}  ({slug})")
                _run_distdiff_for_source(
                    dataset_key    = ds_key,
                    dataset_arg    = ds_key,
                    resolution     = "",
                    is_smile       = True,
                    ds_start       = ds_start,
                    ds_end         = ds_end,
                    window_days    = window_days,
                    data_type      = data_type,
                    slug           = slug,
                    catchment_title= catchment_title,
                    fig_dirs       = DIST_DIFF_FIG_DIRS,
                    dist_diff_combos = DIST_DIFF_COMBOS,
                    bin_width_mm   = BIN_WIDTH_MM,
                    force_recompute= False,
                )

    # ── Reanalysis datasets ───────────────────────────────────────────────────
    for label, ds_arg, res in REANALYSIS_DD:
        from catchment_tools import get_cached_year_range
        avail_start, avail_end = get_cached_year_range(ds_arg, res, window_days)
        if avail_start is None:
            print(f"\n  [skip] No {window_days}-day cache for {label}")
            continue

        for data_type in ("annual_max", "daily"):
            print(f"\n  [REANALYSIS] {label} | {data_type} | {avail_start}–{avail_end}")
            for slug, catchment_title in cfg.CATCHMENTS.items():
                print(f"  Catchment: {catchment_title}  ({slug})")
                _run_distdiff_for_source(
                    dataset_key    = label,
                    dataset_arg    = ds_arg,
                    resolution     = res,
                    is_smile       = False,
                    ds_start       = avail_start,
                    ds_end         = avail_end,
                    window_days    = window_days,
                    data_type      = data_type,
                    slug           = slug,
                    catchment_title= catchment_title,
                    fig_dirs       = DIST_DIFF_FIG_DIRS,
                    dist_diff_combos = DIST_DIFF_COMBOS,
                    bin_width_mm   = BIN_WIDTH_MM,
                    force_recompute= False,
                )

print(f"\n[distdiff] ✓ Done. All figures saved to:")
for d in DIST_DIFF_FIG_DIRS:
    print(f"  {d}")

### Create Q-Q mapping Plots

In [ ]:
# Per-catchment QQ plots: CESM2-LE and GFDL-SPEAR vs. Reanalysis
# Runs for both annual_max and daily data types.

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc, daily_pc in [
        (1, am_1day_per_catchment, daily_1day_per_catchment),
        (2, am_2day_per_catchment, daily_2day_per_catchment),]:
        for data_type, pc in [
            ("annual_max", am_pc),
            ("daily",      daily_pc),]:
            for climate_key in ("cesm2_le", "gfdl_spear_med_le"):
                reanalysis = {k: pc[slug][k] for k in REANALYSIS_KEYS if k in pc[slug]}
                out = [
                    d / f"qq-plot_{data_type}_{climate_key}_{window_days}day_{slug}_{EVAL_PERIOD_TAG}.pdf"
                    for d in FIG_DIRS]
                make_qq_figure(
                    climate_key,
                    pc[slug][climate_key],
                    reanalysis,
                    window_days=window_days,
                    out_paths=out,
                    data_type=data_type,
                    catchment_title=catchment_title,)

### Create Statistical Analysis for QQ-Plot

In [ ]:
# Per-catchment percentile mapping tables

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc in [
        (1, am_1day_per_catchment),
        (2, am_2day_per_catchment),]:
        refs = {k: am_pc[slug][k] for k in REANALYSIS_KEYS if k in am_pc[slug]}

        for climate_key in ("cesm2_le", "gfdl_spear_med_le"):
            df_pct = build_percentile_mapping_table(
                climate_key,
                am_pc[slug][climate_key],
                refs,
                percentiles=PERCENTILES_TO_COMPARE,
            ).round(1)

            print(f"\nPercentile mapping: {climate_key} | {window_days}-day | {slug} | {EVAL_PERIOD_TAG}")
            display(df_pct)

            for d in FIG_DIRS:
                out = d / f"percentile_mapping_{climate_key}_{window_days}day_{slug}_{EVAL_PERIOD_TAG}.csv"
                df_pct.to_csv(out, index=False)
                print(f"Saved -> {out}")

In [ ]:
# Per-catchment distribution summary tables

for slug, catchment_title in cfg.CATCHMENTS.items():
    for window_days, am_pc in [
        (1, am_1day_per_catchment),
        (2, am_2day_per_catchment),]:
        summary = build_distribution_summary_table(am_pc[slug]).round(2)

        print(f"\n{window_days}-day summary | {slug} | {EVAL_PERIOD_TAG}")
        display(summary)

        for d in FIG_DIRS:
            out = d / f"distribution_summary_{window_days}day_{slug}_{EVAL_PERIOD_TAG}.csv"
            summary.to_csv(out, index=False)
            print(f"Saved -> {out}")